In [0]:
from pyspark.sql.functions import col, when, trim, coalesce, lit, row_number, monotonically_increasing_id, try_to_date
from pyspark.sql import Window

# Reading from bronze
df_cust = spark.table("novocart_global.novocart_bronze.customers_raw")

# Deduplicate using business key
window_spec = Window.partitionBy("customer_id").orderBy(monotonically_increasing_id())

df_cust_clean = df_cust.withColumn("row_num", row_number().over(window_spec)) \
             .filter(col("row_num") == 1) \
             .drop("row_num")

# Clean customer_name & email
df_cust_clean = df_cust_clean.withColumn(
    "customer_name",
    when(col("customer_name").isNull() | col("customer_name").isin("NULL", "\\N", "?"), "Unknown Customer")
    .otherwise(trim(col("customer_name")))
).withColumn(
    "email",
    when(col("email").isNull() | col("email").isin("NULL", "\\N", "?"), "no_email@unknown.com")
    .otherwise(trim(col("email")))
)

# Standardize date format
df_cust_clean = df_cust_clean.withColumn(
    "registration_date",
    coalesce(
        try_to_date(col("registration_date"), "yyyy-MM-dd"),
        try_to_date(col("registration_date"), "yyyy/MM/dd"),
        try_to_date(col("registration_date"), "dd/MM/yyyy"),
        lit("1900-01-01").cast("date")
    )
)

#  Final selection and Write
df_silver_cust = df_cust_clean.select(
    "customer_id", "customer_name", "email", 
    "registration_date", "country_code", "channel"
)

df_silver_cust.write.format("delta").mode("overwrite") \
    .saveAsTable("novocart_global.novocart_silver.customers_cleaned")

display(spark.table("novocart_global.novocart_silver.customers_cleaned"))

In [0]:
# Reading from bronze
df_orders = spark.table("novocart_global.novocart_bronze.orders_raw")

# 1. Deduplicate
window_spec = Window.partitionBy("order_id").orderBy(monotonically_increasing_id())
df_orders_clean = df_orders.withColumn("row_num", row_number().over(window_spec)) \
                 .filter(col("row_num") == 1).drop("row_num")

# 2. Standardize order_status (multilingual handling)
df_orders_clean = df_orders_clean.withColumn(
    "order_status",
    when(col("order_status").isin("completed", "已完成", "completado", "abgeschlossen", "पूर्ण"), "completed")
    .when(col("order_status").isin("pending", "pendiente", "待处理", "ausstehend"), "pending")
    .when(col("order_status").isin("cancelled", "cancelado", "storniert", "रद्द"), "cancelled")
    .when(col("order_status").isin("shipped", "भेज दिया"), "shipped")
    .otherwise("unknown")
)

# 3. Handling Dates and Amounts
df_orders_clean = df_orders_clean.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd/MM/yyyy"),
        lit("1900-01-01").cast("date")
    )
).withColumn("total_amount", col("total_amount").cast("double"))

# 4. Final selection and Write
df_silver_orders = df_orders_clean.select(
    "order_id", "customer_id", "order_date", "order_status", 
    "country_code", "channel", "total_amount", "currency"
)

df_silver_orders.write.format("delta").mode("overwrite") \
    .saveAsTable("novocart_global.novocart_silver.orders_cleaned")

display(spark.table("novocart_global.novocart_silver.orders_cleaned"))

In [0]:
from pyspark.sql.functions import expr, round

# Reading from bronze
df_items = spark.table("novocart_global.novocart_bronze.order_items_raw")

# 1. Deduplicate & Type Cast
window_spec = Window.partitionBy("order_item_id").orderBy(monotonically_increasing_id())
df_items_clean = df_items.withColumn("row_num", row_number().over(window_spec)) \
                 .filter(col("row_num") == 1).drop("row_num")

df_items_clean = df_items_clean.withColumn("quantity", coalesce(expr("try_cast(quantity as int)"), lit(0))) \
                               .withColumn("unit_price", coalesce(expr("try_cast(unit_price as double)"), lit(0.0)))

# 2. Calculated Column
df_items_clean = df_items_clean.withColumn("line_total", round(col("quantity") * col("unit_price"), 2))

# 3. Writing to Silver
df_silver_items = df_items_clean.select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "line_total")

df_silver_items.write.format("delta").mode("overwrite") \
    .saveAsTable("novocart_global.novocart_silver.order_items_cleaned")

display(spark.table("novocart_global.novocart_silver.order_items_cleaned"))

In [0]:
from pyspark.sql.functions import col, when, trim, lower, expr, round

# 1. Load Tables 
df_items_raw = spark.table("novocart_global.novocart_bronze.order_items_raw")
df_orders_silver = spark.table("novocart_global.novocart_silver.orders_cleaned")

# 2. Replace noise characters and Normalize keys

df_items_prepped = df_items_raw.replace(["\\N", "?", ""], None) \
    .withColumn("join_key_items", lower(trim(col("order_id"))))

# Preparing the reference table for joining
df_orders_ref = df_orders_silver.select(
    lower(trim(col("order_id"))).alias("join_key_orders")
).distinct() # Distinct ensures we don't accidentally duplicate rows in the join

# 3. Join for Orphan Detection
df_joined = df_items_prepped.join(
    df_orders_ref,
    df_items_prepped.join_key_items == df_orders_ref.join_key_orders,
    "left"
)

# 4. Flagging, Casting, and Logic
df_final = df_joined.withColumn(
    "is_orphan",
    when(col("join_key_orders").isNull(), 1).otherwise(0)
).withColumn(
    "orphan_reason",
    when(col("order_id").isNull(), "Missing Key in Source")
    .when(col("join_key_orders").isNull(), "No Parent Order Found in Silver")
    .otherwise("Valid")
) \
.withColumn("quantity", col("quantity").cast("int")) \
.withColumn("unit_price", col("unit_price").cast("double")) \
.withColumn("line_total", round(col("quantity") * col("unit_price"), 2))

# 5. Select Final Columns & Drop Join Keys
df_silver_order_items = df_final.select(
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_total",
    "is_orphan",
    "orphan_reason"
)

# 6. Write to Silver Layer
df_silver_order_items.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novocart_global.novocart_silver.order_items_cleaned")

# 7. Summary Report
print("Orphan Detection Summary:")
df_silver_order_items.groupBy("is_orphan", "orphan_reason").count().show()

In [0]:
# Reading from bronze
df_ex = spark.table("novocart_global.novocart_bronze.exchange_rates_raw")

# 1. Deduplicate on Composite Key
window_spec = Window.partitionBy("currency_code", "effective_date").orderBy(monotonically_increasing_id())
df_ex_clean = df_ex.withColumn("row_num", row_number().over(window_spec)) \
                 .filter(col("row_num") == 1).drop("row_num")

# 2. Fixing Types
df_ex_clean = df_ex_clean.withColumn("exchange_rate_to_usd", col("exchange_rate_to_usd").cast("double")) \
                         .withColumn("effective_date", try_to_date(col("effective_date"), "yyyy-MM-dd"))

# 3. Write to Silver
df_silver_ex = df_ex_clean.select("currency_code", "exchange_rate_to_usd", "effective_date")

df_silver_ex.write.format("delta").mode("overwrite") \
    .saveAsTable("novocart_global.novocart_silver.exchange_rates_cleaned")

display(spark.table("novocart_global.novocart_silver.exchange_rates_cleaned"))

In [0]:
for tbl in [
    "customers_cleaned",
    "orders_cleaned",
    "order_items_cleaned",
    "exchange_rates_cleaned",
    "products_cleaned"
]:
    print(f"Schema for novocart_global.novocart_silver.{tbl}:")
    spark.table(f"novocart_global.novocart_silver.{tbl}").printSchema()